# Train a CARE model on real training data

This notebook trains a CARE denoising model from paired training patches (`.npz`) generated in the patch-generation notebook.

### Typical workflow
1. Generate training patches in the datagen notebook
2. Choose the patch file to train on
3. Configure training
4. Train the CARE model
5. Inspect validation predictions
6. Save training metadata

In [ ]:
from pathlib import Path

from ISS_CARE.ISS_CARE_training import (
    build_care_config,
    build_training_metadata,
    configure_tensorflow_memory_growth,
    create_care_model,
    load_care_training_data,
    normalize_patch_dataset,
    plot_patch_examples,
    plot_training_curves,
    predict_on_validation_examples,
    print_model_save_info,
    resolve_patch_file,
    save_training_metadata,
    select_gpu_with_lowest_memory,
    train_care_model,
)

## GPU setup

This notebook is intended for shared GPU servers.

It:
- selects one GPU with the lowest current memory usage
- restricts TensorFlow to that GPU
- enables TensorFlow memory growth so memory is allocated only as needed

In [ ]:
selected_gpu = select_gpu_with_lowest_memory()
gpus = configure_tensorflow_memory_growth()

## User settings

Edit these parameters to match your dataset and training target.

### Dataset / patch file
- `CARE_ROOT`: root folder containing your CARE data
- `PATCH_FILE_NAME`: patch file to train on
- `MODEL_DIRNAME`: folder under `CARE_ROOT` where trained models are saved
- `MODEL_NAME`: name of this training run

### Validation
- `VALIDATION_SPLIT`: fraction of patches reserved for validation

### Training
- `TRAIN_BATCH_SIZE`
- `TRAIN_STEPS_PER_EPOCH`
- `TRAIN_EPOCHS`

### Model
- `UNET_KERN_SIZE`
- `TRAIN_LEARNING_RATE`
- `PROBABILISTIC`

In [ ]:
from pathlib import Path

# ----------------------------
# User settings
# ----------------------------

HOME = Path.home()
CARE_ROOT = HOME / "moldia-archive" / "CARE_training_data"
CARE_ROOT = CARE_ROOT.expanduser().resolve()

# Patch file to train on
PATCH_FILE_NAME = "ALL_SAMPLES__NON_DAPI__train_patches.npz"
# PATCH_FILE_NAME = "ALL_SAMPLES__DAPI_ONLY__train_patches.npz"

# Model output
MODEL_DIRNAME = "care_models"
MODEL_DIR = CARE_ROOT / MODEL_DIRNAME
MODEL_NAME = "care_non_dapi"
# MODEL_NAME = "care_dapi_only"

# Validation split
VALIDATION_SPLIT = 0.05

# Training parameters
TRAIN_BATCH_SIZE = 8
TRAIN_STEPS_PER_EPOCH = 100
TRAIN_EPOCHS = 100

# Model parameters
UNET_KERN_SIZE = 3
TRAIN_LEARNING_RATE = 2e-4
PROBABILISTIC = False

print("CARE_ROOT:", CARE_ROOT)
print("PATCH_FILE_NAME:", PATCH_FILE_NAME)
print("MODEL_DIR:", MODEL_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("VALIDATION_SPLIT:", VALIDATION_SPLIT)
print("TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)
print("TRAIN_STEPS_PER_EPOCH:", TRAIN_STEPS_PER_EPOCH)
print("TRAIN_EPOCHS:", TRAIN_EPOCHS)
print("UNET_KERN_SIZE:", UNET_KERN_SIZE)
print("TRAIN_LEARNING_RATE:", TRAIN_LEARNING_RATE)
print("PROBABILISTIC:", PROBABILISTIC)

## Resolve the training patch file

This cell finds the patch directory and resolves the `.npz` patch file that will be used for training.

In [ ]:
PATCH_DIR, PATCH_FILE = resolve_patch_file(
    care_root=CARE_ROOT,
    patch_file_name=PATCH_FILE_NAME,
)

print("PATCH_DIR:", PATCH_DIR)
print("PATCH_FILE:", PATCH_FILE)

## Load training data

This cell loads the selected patch file and splits it into:
- training data
- validation data

It also reads the patch axes and determines the number of input and output channels needed for the CARE model.

In [ ]:
X, Y, X_val, Y_val, axes, n_channel_in, n_channel_out = load_care_training_data(
    patch_file=PATCH_FILE,
    validation_split=VALIDATION_SPLIT,
)

X, Y, X_val, Y_val = normalize_patch_dataset(
    X,
    Y,
    X_val,
    Y_val,
    enabled=True,
    pmin=1.0,
    pmax=99.8,
    eps=1e-8,
)

## Inspect a few example patches

This is a quick sanity check to confirm that:
- source and target patches match
- intensities look reasonable
- the selected patch file is the one you intended to train on

In [ ]:
plot_patch_examples(
    X,
    Y,
    n_show=5,
    title="Training patch pairs (top: input, bottom: target)",
)

## Build CARE configuration

This creates the CARE training configuration from:
- patch axes
- input channels
- output channels
- training hyperparameters

In [ ]:
config = build_care_config(
    axes=axes,
    n_channel_in=n_channel_in,
    n_channel_out=n_channel_out,
    train_batch_size=TRAIN_BATCH_SIZE,
    train_steps_per_epoch=TRAIN_STEPS_PER_EPOCH,
    train_epochs=TRAIN_EPOCHS,
    unet_kern_size=UNET_KERN_SIZE,
    train_learning_rate=TRAIN_LEARNING_RATE,
    probabilistic=PROBABILISTIC,
)

## Create the CARE model

The model will be saved in:

`MODEL_DIR / MODEL_NAME /`

In [ ]:
model = create_care_model(
    config=config,
    model_name=MODEL_NAME,
    model_dir=MODEL_DIR,
)

print_model_save_info(
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME,
)

print("Model output directory:", (MODEL_DIR / MODEL_NAME).resolve())

## Inspect the model architecture

In [ ]:
model.keras_model._name = MODEL_NAME
model.keras_model.summary()

## Train the model

This can take from minutes to hours depending on:
- dataset size
- patch size
- GPU availability
- training settings

In [ ]:
history = train_care_model(
    model=model,
    X=X,
    Y=Y,
    X_val=X_val,
    Y_val=Y_val,
)

print("Training finished.")

## Plot training history

In [ ]:
plot_training_curves(history)

## Quick prediction check on validation patches

This gives a fast qualitative check before applying the model to larger images.

In [ ]:
X_example, Y_example, Y_pred = predict_on_validation_examples(
    model=model,
    X_val=X_val,
    Y_val=Y_val,
    probabilistic=PROBABILISTIC,
    n_examples=5,
)

## Save training metadata

This saves a metadata file describing:
- dataset path
- patch file used
- model name and output directory
- training parameters
- TensorFlow version
- selected GPU
- patch/data shapes

In [ ]:
metadata = build_training_metadata(
    care_root=CARE_ROOT,
    patch_dir=PATCH_DIR,
    patch_file=PATCH_FILE,
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME,
    validation_split=VALIDATION_SPLIT,
    train_batch_size=TRAIN_BATCH_SIZE,
    train_steps_per_epoch=TRAIN_STEPS_PER_EPOCH,
    train_epochs=TRAIN_EPOCHS,
    unet_kern_size=UNET_KERN_SIZE,
    train_learning_rate=TRAIN_LEARNING_RATE,
    probabilistic=PROBABILISTIC,
    axes=axes,
    X_shape=X.shape,
    Y_shape=Y.shape,
    X_val_shape=X_val.shape,
    Y_val_shape=Y_val.shape,
    n_channel_in=n_channel_in,
    n_channel_out=n_channel_out,
    normalization_enabled=True,
    normalization_pmin=1.0,
    normalization_pmax=99.8,
    normalization_eps=1e-8,
)

save_training_metadata(
    metadata,
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME,
)

## Load the model later

To load the trained model in another notebook:

```python
from csbdeep.models import CARE

model = CARE(
    config=None,
    name=MODEL_NAME,
    basedir=str(MODEL_DIR),
)